In [1]:
import pandas as pd
import numpy as np
import torch

In [2]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
    
print(device)

cuda


In [3]:
df = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")
df.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
from torch.utils.data import random_split

train_size = int(0.9 * len(df)) 
train_df, valid_df = df[:train_size], df[train_size:]
print(len(train_df))
print(len(valid_df))

37800
4200


In [5]:
valid_df = valid_df.reset_index().drop(columns = ["index"])

In [6]:
test_df = pd.read_csv("/kaggle/input/digit-recognizer/test.csv")
test_df.head()

,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
idx = 2
sample = df.loc[idx]
label = int(sample["label"])
pixels = np.array(sample.drop("label")).reshape(28, 28)

print(label)
print(pixels.shape)

1
(28, 28)


In [8]:
from torch.utils.data import Dataset

class MNISTDataset(Dataset):
    def __init__(self, df, mode="train"):
        self.mode = mode
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self ,idx):
        sample = self.df.loc[idx]
        if self.mode == "train":
            label = int(sample["label"])
            image = np.array(sample.drop("label")).reshape(1, 28, 28)
            image = torch.Tensor(image) / 255.0
            return image, label
        else:
            image = np.array(sample).reshape(1, 28, 28)
            image = torch.Tensor(image) / 255.0
            return image

In [9]:
train_set = MNISTDataset(train_df, mode="train")
valid_set = MNISTDataset(valid_df, mode="train")
test_set = MNISTDataset(test_df, mode="test")

In [10]:
print(f"Size of Train set : {len(train_set)}")
print(f"Size of Valid set : {len(valid_set)}")
print(f"Size of Test set : {len(test_set)}")

Size of Train set : 37800
Size of Valid set : 4200
Size of Test set : 28000


In [11]:
from torch.utils.data import DataLoader
BATCH_SIZE = 64
train_loader = DataLoader(train_set, batch_size = BATCH_SIZE, shuffle = True)
valid_loader = DataLoader(valid_set, batch_size = BATCH_SIZE, shuffle = True)
test_loader = DataLoader(test_set, batch_size = BATCH_SIZE, shuffle = False)

In [12]:
print(f"Total no. of batches in trainloader: {len(train_loader)}")
print(f"Total no. of batches in testloader: {len(valid_loader)}")


Total no. of batches in trainloader: 591
Total no. of batches in testloader: 66


In [13]:
for image, label in train_loader:
    break
print (f'One batch image shape: {image.shape}')
print (f'Label: {label.shape}')

for image in test_loader:
    break
print (f'One batch image shape: {image.shape}')

One batch image shape: torch.Size([64, 1, 28, 28])
Label: torch.Size([64])
One batch image shape: torch.Size([64, 1, 28, 28])


In [14]:
from torch import nn

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride = 1, downsample = None):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Sequential(
                        nn.Conv2d(in_channels, out_channels, kernel_size = 3, stride = stride, padding = 1),
                        nn.BatchNorm2d(out_channels),
                        nn.ReLU())
        self.conv2 = nn.Sequential(
                        nn.Conv2d(out_channels, out_channels, kernel_size = 3, stride = 1, padding = 1),
                        nn.BatchNorm2d(out_channels))
        self.downsample = downsample
        self.relu = nn.ReLU()
        self.out_channels = out_channels
        
    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.conv2(out)
        if self.downsample:
            residual = self.downsample(x)
        out += residual
        out = self.relu(out)
        return out

In [15]:
class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes = 10):
        super(ResNet, self).__init__()
        self.inplanes = 64
        self.conv1 = nn.Sequential(
                        nn.Conv2d(1, 64, kernel_size = 7, stride = 2, padding = 3),
                        nn.BatchNorm2d(64),
                        nn.ReLU())
        self.maxpool = nn.MaxPool2d(kernel_size = 3, stride = 2, padding = 1)
        self.layer0 = self._make_layer(block, 64, layers[0], stride = 1)
        self.layer1 = self._make_layer(block, 128, layers[1], stride = 2)
        self.layer2 = self._make_layer(block, 256, layers[2], stride = 2)
        self.layer3 = self._make_layer(block, 512, layers[3], stride = 2)
        self.fc = nn.Linear(512, num_classes)
        
    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes:
            
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes, kernel_size=1, stride=stride),
                nn.BatchNorm2d(planes),
            )
        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)
    
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.maxpool(x)
        x = self.layer0(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = x.view(x.size(0), -1)
        x = self.fc(x)

        return x

In [16]:
num_classes = 10
LEARNING_RATE = 0.01

model = ResNet(ResidualBlock, [1, 1, 1, 1]).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE, weight_decay = 0.001, momentum = 0.9)  

total_step = len(train_loader)
print(total_step)

591


In [17]:
from tqdm import tqdm 

EPOCHS = 5
for epoch in range(EPOCHS):
    
    #Training
    for images, labels in tqdm(train_loader):
        
        images = images.to(device)
        labels = labels.to(device)
        
        preds = model(images)
        loss = criterion(preds, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Evaluation
    with torch.no_grad():
        correct = 0
        total = 0
        for images, labels in valid_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
        print('Accuracy of the network on the {} validation images: {} %'.format(5000, 100 * correct / total)) 

100%|██████████| 591/591 [00:40<00:00, 14.73it/s]


Accuracy of the network on the 5000 validation images: 97.76190476190476 %


100%|██████████| 591/591 [00:28<00:00, 20.44it/s]


Accuracy of the network on the 5000 validation images: 98.19047619047619 %


100%|██████████| 591/591 [00:28<00:00, 20.60it/s]


Accuracy of the network on the 5000 validation images: 98.54761904761905 %


100%|██████████| 591/591 [00:28<00:00, 20.55it/s]


Accuracy of the network on the 5000 validation images: 98.71428571428571 %


100%|██████████| 591/591 [00:28<00:00, 20.52it/s]


Accuracy of the network on the 5000 validation images: 98.76190476190476 %


In [18]:
answer = []
with torch.no_grad():
    for images in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        answer += [int(p) for p in predicted]
print(answer[0])
print(len(answer))

2
28000


In [19]:
submission = pd.DataFrame(answer)
submission = submission.reset_index()
submission["index"] += 1
submission.columns = ["ImageId", "Label"]
submission = submission.set_index("ImageId")
submission.head()

,Label
ImageId,
1,2
2,0
3,9
4,9
5,3


In [20]:
submission.to_csv("submission.csv")